In [0]:
-- Create a bronze demo table if missing
CREATE TABLE IF NOT EXISTS bronze_demo.accounts_raw AS
SELECT * FROM VALUES
  ('A001','Acme Inc','US'),
  ('A002','Globex','CA'),
  ('A003','Initech',NULL)
AS T(AccountId, AccountName, Country);


In [0]:
SELECT current_catalog(), current_schema();
SHOW CATALOGS;


In [0]:
USE CATALOG main;
CREATE SCHEMA IF NOT EXISTS bronze_demo;


In [0]:
SELECT current_catalog(), current_schema();


In [0]:
USE CATALOG spark_catalog;  -- ok to run; it just sets the session


In [0]:
CREATE SCHEMA IF NOT EXISTS bronze_demo;
USE SCHEMA bronze_demo;

In [0]:
CREATE OR REPLACE TABLE accounts_raw AS
SELECT * FROM VALUES
  ('A001','Acme Inc','US'),
  ('A002','Globex','CA'),
  ('A003','Initech',NULL)
AS T(AccountId, AccountName, Country);

In [0]:
SELECT * FROM bronze_demo.accounts_raw;   -- works because you’re in spark_catalog

In [0]:
-- Create a Silver schema in spark_catalog (2-part names only)
CREATE SCHEMA IF NOT EXISTS silver_demo;

-- Build the Silver table from Bronze
CREATE OR REPLACE TABLE silver_demo.accounts_clean AS
WITH src AS (
  SELECT
    CAST(AccountId AS STRING)         AS AccountId,
    TRIM(AccountName)                 AS AccountName,
    UPPER(COALESCE(Country,'Unknown')) AS Country,
    ROW_NUMBER() OVER (
      PARTITION BY AccountId
      ORDER BY AccountId
    ) AS rn
  FROM bronze_demo.accounts_raw
)
SELECT
  AccountId,
  AccountName,
  Country
FROM src
WHERE rn = 1;   -- dedupe


In [0]:
-- Create a Gold schema in spark_catalog
CREATE SCHEMA IF NOT EXISTS gold_demo;

-- Build a simple KPI table
CREATE OR REPLACE TABLE gold_demo.accounts_by_country AS
SELECT
  Country,
  COUNT(*) AS AccountCount
FROM silver_demo.accounts_clean
GROUP BY Country;


In [0]:
-- Optimization (works for Delta tables)
OPTIMIZE silver_demo.accounts_clean;
OPTIMIZE gold_demo.accounts_by_country;

-- Change history (Delta)
DESCRIBE HISTORY silver_demo.accounts_clean;
DESCRIBE HISTORY gold_demo.accounts_by_country;


In [0]:
SHOW CATALOGS;
  SHOW SCHEMAS IN catalog_name;
  SHOW TABLES IN schema_name;